# TabPFN timing / memory benchmark  (without_HPO | with_HPO | RF+TabPFN)

Reproduces each variant's construction from the `*_Calc` notebooks and measures, per
**(variant x target)**:

| metric | how |
|---|---|
| Training wall time [s] | `perf_counter` around `.fit` |
| Training GPU time [s]  | `cuda.Event` elapsed across `.fit` |
| Training CPU time [s]  | `process_time` around `.fit` |
| Per-sample inference [ms] | `predict` on the test split / n_test |
| Used RAM [MB] | peak process RSS rise during `.fit` |
| GPU memory [MB] | `torch.cuda.max_memory_allocated` |
| Model size [MB] | `joblib.dump` size |

Runs **in-process**, one combo at a time, freeing each model (`del`+`gc`+`empty_cache`)
before the next so RAM/GPU-mem stay per-combo. Run on an **H100** with the `py_a6` kernel.
With `N_TRIALS=100`, the `with_HPO` and `RF+TabPFN` rows are the slow part (hours total).

In [1]:
import os, gc, time, tempfile, threading
import numpy as np, pandas as pd

# thread env (fidelity with the SLURM runs) — set BEFORE importing torch/tabpfn
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = str(n_cpus)
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

import torch, joblib, psutil
from sklearn.model_selection import train_test_split
try: torch.set_num_threads(n_cpus)
except Exception: pass
CUDA = torch.cuda.is_available()
print("CUDA:", CUDA, (torch.cuda.get_device_name(0) if CUDA else ""))

CUDA: True NVIDIA H100


In [2]:
N_TRIALS = 100          # with_HPO TPE trials (matches the runs)
TARGETS  = ["P_bubble", "P_dew", "gamma", "interfacial_thickness"]
VARIANTS = ["without_HPO", "with_HPO", "RF_TabPFN"]
OUTDIR   = "BENCHMARK_TIMING"

# per-(variant, target) seed, copied from each *_Calc notebook
SEEDS = {
    "without_HPO": {"P_bubble": 50015,  "P_dew": 855015,  "gamma": 50005,  "interfacial_thickness": 655552},
    "with_HPO":    {"P_bubble": 454015, "P_dew": 6702315, "gamma": 844015, "interfacial_thickness": 655552},
    "RF_TabPFN":   {"P_bubble": 50015,  "P_dew": 855015,  "gamma": 50005,  "interfacial_thickness": 655552},
}
DATA_CANDIDATES = [
    os.environ.get("TABPFN_DATASET", ""),
    "/gpfs/home6/draju/A6/DATASET_A4/interfacial_results_dataset_A4.csv",
    "/home/darshan/A6/PCSAFT_cDFT/PART_1/interfacial_results_dataset_A4.csv",
]

In [3]:
def dataset_path():
    for p in DATA_CANDIDATES:
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError(f"dataset not found in {DATA_CANDIDATES}")

def load_split(target, seed):
    df = pd.read_csv(dataset_path())
    z = [c for c in df.columns if c.startswith("z_") and (df[c] != 0).any()]
    features = ["temperature", "pressure"] + z
    X, y = df[features], df[target]
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, random_state=seed)
    X_te, X_va, y_te, y_va   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=seed)
    return X_tr, y_tr, X_te, y_te

def build_model(variant, seed):
    from tabpfn import TabPFNRegressor
    if variant == "without_HPO":
        return TabPFNRegressor(random_state=seed, ignore_pretraining_limits=True,
                               fit_mode="fit_preprocessors")
    if variant == "RF_TabPFN":
        from tabpfn_extensions.rf_pfn import RandomForestTabPFNRegressor
        base = TabPFNRegressor(random_state=seed, ignore_pretraining_limits=True,
                               fit_mode="fit_preprocessors")
        return RandomForestTabPFNRegressor(tabpfn=base, n_estimators=10, max_depth=3)
    if variant == "with_HPO":
        from tabpfn_extensions.hpo import TunedTabPFNRegressor
        from tabpfn_extensions.hpo.search_space import get_param_grid_hyperopt
        from tabpfn.constants import ModelVersion
        ss = get_param_grid_hyperopt("regression", model_version=ModelVersion.V2_5)
        ss["ignore_pretraining_limits"] = True
        return TunedTabPFNRegressor(n_trials=N_TRIALS, metric="rmse", n_validation_size=0.2,
                                    shuffle_data=True, search_algorithm_type="tpe", device="auto",
                                    random_state=seed, verbose=False, search_space=ss)
    raise ValueError(variant)

class RSSPeak:
    # peak process RSS (MB) rise over the with-block
    def __init__(self, dt=0.05):
        self.p = psutil.Process(); self.dt = dt
        self.base = self.p.memory_info().rss; self.peak = self.base
    def __enter__(self):
        self._run = True
        self.t = threading.Thread(target=self._loop, daemon=True); self.t.start(); return self
    def _loop(self):
        while self._run:
            self.peak = max(self.peak, self.p.memory_info().rss); time.sleep(self.dt)
    def __exit__(self, *a):
        self._run = False; self.t.join(timeout=1.0)
    @property
    def peak_mb(self): return (self.peak - self.base) / 1e6

In [4]:
def run_one(variant, target):
    seed = SEEDS[variant][target]
    X_tr, y_tr, X_te, y_te = load_split(target, seed)
    model = build_model(variant, seed)

    if CUDA:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        ev0, ev1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)

    with RSSPeak() as rss:                       # ---- training ----
        if CUDA: ev0.record()
        tw0, tc0 = time.perf_counter(), time.process_time()
        model.fit(X_tr, y_tr)
        if CUDA: torch.cuda.synchronize()
        train_wall, train_cpu = time.perf_counter() - tw0, time.process_time() - tc0
        if CUDA: ev1.record(); torch.cuda.synchronize()
    train_gpu = ev0.elapsed_time(ev1) / 1e3 if CUDA else float("nan")
    gpu_mem   = torch.cuda.max_memory_allocated() / 1e6 if CUDA else float("nan")

    t0 = time.perf_counter()                     # ---- inference ----
    model.predict(X_te)
    if CUDA: torch.cuda.synchronize()
    ips = (time.perf_counter() - t0) / len(X_te)

    with tempfile.NamedTemporaryFile(suffix=".joblib") as f:   # ---- size ----
        joblib.dump(model, f.name); size_mb = os.path.getsize(f.name) / 1e6

    row = dict(variant=variant, target=target, seed=seed,
               n_train=len(X_tr), n_test=len(X_te),
               train_wall_s=round(train_wall, 4), train_gpu_s=round(train_gpu, 4),
               train_cpu_s=round(train_cpu, 4), infer_per_sample_ms=ips * 1e3,
               used_ram_mb=round(rss.peak_mb, 4), gpu_mem_mb=round(gpu_mem, 4),
               model_size_mb=round(size_mb, 4))
    del model; gc.collect()
    if CUDA: torch.cuda.empty_cache()
    return row

In [5]:
rows = []
for variant in VARIANTS:
    for target in TARGETS:
        print(f"=== {variant} / {target} ===", flush=True)
        try:
            r = run_one(variant, target); rows.append(r)
            print(f"  wall={r['train_wall_s']}s gpu={r['train_gpu_s']}s "
                  f"infer={r['infer_per_sample_ms']:.4f}ms ram={r['used_ram_mb']}MB "
                  f"gpumem={r['gpu_mem_mb']}MB size={r['model_size_mb']}MB", flush=True)
        except Exception:
            import traceback; traceback.print_exc()

os.makedirs(OUTDIR, exist_ok=True)
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUTDIR, "benchmark_timing.csv"), index=False)
df

=== without_HPO / P_bubble ===


  wall=1.6286s gpu=1.6287s infer=1.6985ms ram=364.6095MB gpumem=51.5205MB size=87.8564MB


=== without_HPO / P_dew ===


  wall=1.3412s gpu=1.3413s infer=0.7318ms ram=30.4128MB gpumem=86.1235MB size=87.8565MB


=== without_HPO / gamma ===


...

=== RF_TabPFN / gamma ===


  wall=0.0738s gpu=0.0739s infer=260.1805ms ram=0.0MB gpumem=34.603MB size=157.4337MB


=== RF_TabPFN / interfacial_thickness ===


  wall=0.0746s gpu=0.0747s infer=259.8030ms ram=0.0MB gpumem=34.603MB size=154.0955MB


,variant,target,seed,n_train,n_test,train_wall_s,train_gpu_s,train_cpu_s,infer_per_sample_ms,used_ram_mb,gpu_mem_mb,model_size_mb
0,without_HPO,P_bubble,50015,13552,2904,1.6286,1.6287,10.6406,1.698456,364.6095,51.5205,87.8564
1,without_HPO,P_dew,855015,13552,2904,1.3412,1.3413,9.3378,0.731838,30.4128,86.1235,87.8565
2,without_HPO,gamma,50005,13552,2904,1.3075,1.3076,8.9385,0.733553,27.6316,86.1235,87.8565
3,without_HPO,interfacial_thickness,655552,13552,2904,1.5304,1.5305,11.2698,0.735898,32.0020,86.1235,87.8565
4,with_HPO,P_bubble,454015,13552,2904,860.0724,860.0757,1572.4070,0.452605,4348.5348,4853.8066,50.5107
5,with_HPO,P_dew,6702315,13552,2904,1429.7212,1429.7269,3267.9973,5.438467,3508.6623,4977.3041,51.0857
6,with_HPO,gamma,844015,13552,2904,1217.4278,1217.4326,2790.7291,1.735462,3251.8799,4853.7958,50.3514
7,with_HPO,interfacial_thickness,655552,13552,2904,524.6616,524.6636,1190.2044,0.485773,2137.2969,4896.6036,52.6012
8,RF_TabPFN,P_bubble,50015,13552,2904,0.0859,0.0860,0.0844,270.531637,0.0000,34.6030,154.2924
9,RF_TabPFN,P_dew,855015,13552,2904,0.0723,0.0724,0.0719,276.871927,0.0000,34.6030,153.4447


In [6]:
# ---- Table-4-style tables: rows = metrics, columns = variants (one per target) ----
ROWS = [("Training wall time [s]",      "train_wall_s"),
        ("Training GPU time [s]",       "train_gpu_s"),
        ("Per-sample inference [ms]",   "infer_per_sample_ms"),
        ("Used RAM [MB]",               "used_ram_mb"),
        ("GPU memory [MB]",             "gpu_mem_mb"),
        ("Model size [MB]",             "model_size_mb")]

def table_for(target):
    sub = df[df.target == target].set_index("variant")
    data = {lab: [sub.loc[v, key] if v in sub.index else np.nan for v in VARIANTS]
            for lab, key in ROWS}
    return pd.DataFrame(data, index=VARIANTS).T

with open(os.path.join(OUTDIR, "timing_tables.tex"), "w") as f:
    for tgt in TARGETS:
        t = table_for(tgt)
        print(f"\n### {tgt}"); display(t)
        f.write(t.to_latex(float_format="%.4f", caption=f"TabPFN timing/memory --- {tgt}",
                           label=f"tab:timing_{tgt}"))
        f.write("\n")
print("wrote", os.path.join(OUTDIR, "timing_tables.tex"))


### P_bubble


,without_HPO,with_HPO,RF_TabPFN
Training wall time [s],1.628600,860.072400,0.085900
Training GPU time [s],1.628700,860.075700,0.086000
Per-sample inference [ms],1.698456,0.452605,270.531637
Used RAM [MB],364.609500,4348.534800,0.000000
GPU memory [MB],51.520500,4853.806600,34.603000
Model size [MB],87.856400,50.510700,154.292400



### P_dew


,without_HPO,with_HPO,RF_TabPFN
Training wall time [s],1.341200,1429.721200,0.072300
Training GPU time [s],1.341300,1429.726900,0.072400
Per-sample inference [ms],0.731838,5.438467,276.871927
Used RAM [MB],30.412800,3508.662300,0.000000
GPU memory [MB],86.123500,4977.304100,34.603000
Model size [MB],87.856500,51.085700,153.444700



### gamma


,without_HPO,with_HPO,RF_TabPFN
Training wall time [s],1.307500,1217.427800,0.073800
Training GPU time [s],1.307600,1217.432600,0.073900
Per-sample inference [ms],0.733553,1.735462,260.180508
Used RAM [MB],27.631600,3251.879900,0.000000
GPU memory [MB],86.123500,4853.795800,34.603000
Model size [MB],87.856500,50.351400,157.433700



### interfacial_thickness


,without_HPO,with_HPO,RF_TabPFN
Training wall time [s],1.530400,524.661600,0.074600
Training GPU time [s],1.530500,524.663600,0.074700
Per-sample inference [ms],0.735898,0.485773,259.803009
Used RAM [MB],32.002000,2137.296900,0.000000
GPU memory [MB],86.123500,4896.603600,34.603000
Model size [MB],87.856500,52.601200,154.095500


wrote BENCHMARK_TIMING/timing_tables.tex
